In [1]:
import os
from dotenv import load_dotenv

from crewai import Agent, Task, Crew
from crewai import LLM
from crewai.tools import BaseTool
from typing import Type

from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_openai import ChatOpenAI

from pydantic import BaseModel

In [2]:
print('\nStarting the work of the Agents Team based on generative AI\n')

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("There was no API key found.")

OPENAI_API_KEY=api_key
print('\nAgents API keys were successfully loaded!\n')


Starting the work of the Agents Team based on generative AI


Agents API keys were successfully loaded!



In [3]:
class SearchInput(BaseModel):
    query: str


class DuckDuckGoTool(BaseTool):
    name: str = "DuckDuckGo Search"
    description: str = "Executes web search using DuckDuckGo"
    args_schema: Type[BaseModel] = SearchInput 

    def _run(self, query: str) -> str:
        return DuckDuckGoSearchRun().run(query)


class WikipediaTool(BaseTool):
    name: str = "Wikipedia Search"
    description: str = "Search contents on Wikipedia"
    args_schema: Type[BaseModel] = SearchInput

    def _run(self, query: str) -> str:
        wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
        return wiki.run(query)


# Instantiate tools
search_tool = DuckDuckGoTool()
wikipedia_tool = WikipediaTool()

In [ ]:
llm_researcher = LLM(model="gpt-4o", temperature=0.7)
llm_legal_assistant = LLM(model="gpt-4o", temperature=0.2)
llm_reporter = LLM(model="gpt-4o", temperature=0.5)

In [6]:
# Creating the Researcher Agent
researcher = Agent(
    role='Senior Researcher Analyst',
    goal='Do complete searches and collect relevant data',
    backstory="""Você é um analista de pesquisa sênior com 15 anos de experiência em análise de dados. 
    You are a senior researcher analyst with fifteen years of experience in data analysis and market reasearch. 
    You have a postgraduation in Data Science and you are a specialist in recognize patterns and trend analysis. 
    You are great in:
    1. Collect and validate informations from different sources
    2. Identify trends and emergents patterns
    3. Critical evaluation of the quality and relevancy of the data
    4. Provide context and insights for complex topics""",
    verbose=True,
    tools=[search_tool, wikipedia_tool],  
    allow_delegation=True,
    llm=llm_researcher
)


# Creating the Legal Asistent Agent
legal_assistent = Agent(
    role='Legal Assistent',
    goal='Process and transform the data in useful format',
    backstory="""You are a senior legal assistent with experience in the law area,
    you are specialized in the legal standards of the USA. 
    Your scpecialities include:
    1. Preparing and organizing legal documents, such as contracts and petitions
    2. Doing legal researches to support lawyers in specific cases
    3. Following procedural deadlines and to managing the calendar of the office tasks  
    4. Rewrite legal opions under lawyers supervision
    You always guarantee the data integrity when you prepare them to analysis.""",
    verbose=True,
    allow_delegation=True,
    llm=llm_legal_assistant 
)